In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pickle

DATA_DIR = Path("../data/raw/lobster_samples")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

PRICE_SCALE = 10_000
N_LEVELS = 10
TICKERS = ["AAPL", "AMZN", "GOOG", "INTC", "MSFT"]
DATE = "2012-06-21"

# Copy load_lobster from EDA notebook
def load_lobster(ticker, date, data_dir, n_levels=10):
    base = f"{ticker}_{date}_34200000_57600000"
    msg_cols = ["time", "event_type", "order_id", "size", "price", "direction"]
    msg = pd.read_csv(data_dir / f"{base}_message_{n_levels}.csv",
                      header=None, names=msg_cols)
    msg["price"] = msg["price"] / PRICE_SCALE

    ob_cols = []
    for i in range(1, n_levels + 1):
        ob_cols += [f"ask_price_{i}", f"ask_size_{i}",
                    f"bid_price_{i}", f"bid_size_{i}"]
    ob = pd.read_csv(data_dir / f"{base}_orderbook_{n_levels}.csv",
                     header=None, names=ob_cols)
    price_cols = [c for c in ob_cols if "price" in c]
    ob[price_cols] = ob[price_cols] / PRICE_SCALE

    df = pd.concat([msg.reset_index(drop=True), ob.reset_index(drop=True)], axis=1)
    df["mid_price"] = (df["ask_price_1"] + df["bid_price_1"]) / 2
    df["spread"]    = df["ask_price_1"] - df["bid_price_1"]
    df["ticker"]    = ticker
    return df

data = {t: load_lobster(t, DATE, DATA_DIR) for t in TICKERS}
print("Loaded:", {t: len(df) for t, df in data.items()})

Loaded: {'AAPL': 400391, 'AMZN': 269748, 'GOOG': 147916, 'INTC': 624040, 'MSFT': 668765}


In [2]:
def make_labels(mid_price: pd.Series, k: int = 50, alpha: float = 0.0):
    """
    Smooth future mid-price over k events, compare to current.
    Returns integer labels: 0=down, 1=stationary, 2=up
    
    k=50 is a reasonable default for ~1-5 second horizon at this event rate.
    """
    m_plus = mid_price.shift(-1).rolling(window=k, min_periods=k).mean().shift(-(k-1))
    
    label = np.where(m_plus > mid_price * (1 + alpha),  2,   # up
            np.where(m_plus < mid_price * (1 - alpha),  0,   # down
                                                         1))  # stationary
    return pd.Series(label, index=mid_price.index)

# Try k=50, check class balance
print(f"Label distribution at k=50, alpha=0.0:\n")
for ticker in TICKERS:
    labels = make_labels(data[ticker]["mid_price"], k=50, alpha=0.0)
    counts = labels.value_counts().sort_index()
    total  = len(labels.dropna())
    pcts   = (counts / total * 100).round(1)
    print(f"  {ticker}: down={pcts.get(0,0)}%  stat={pcts.get(1,0)}%  up={pcts.get(2,0)}%")

Label distribution at k=50, alpha=0.0:

  AAPL: down=50.4%  stat=2.2%  up=47.4%
  AMZN: down=48.7%  stat=5.5%  up=45.7%
  GOOG: down=52.9%  stat=2.3%  up=44.8%
  INTC: down=10.0%  stat=80.7%  up=9.4%
  MSFT: down=12.4%  stat=76.5%  up=11.1%


In [3]:
print("Class balance vs alpha threshold:\n")
alphas = [0.0, 1e-5, 2e-5, 5e-5, 1e-4]

for alpha in alphas:
    print(f"  alpha={alpha:.0e}")
    for ticker in ["AAPL", "INTC"]:  # one wide-spread, one tick-constrained
        labels = make_labels(data[ticker]["mid_price"], k=50, alpha=alpha)
        counts = labels.value_counts().sort_index()
        total  = counts.sum()
        pcts   = (counts / total * 100).round(1)
        print(f"    {ticker}: down={pcts.get(0,0)}%  stat={pcts.get(1,0)}%  up={pcts.get(2,0)}%")
    print()

Class balance vs alpha threshold:

  alpha=0e+00
    AAPL: down=50.4%  stat=2.2%  up=47.4%
    INTC: down=10.0%  stat=80.7%  up=9.4%

  alpha=1e-05
    AAPL: down=40.2%  stat=22.0%  up=37.9%
    INTC: down=9.6%  stat=81.4%  up=9.0%

  alpha=2e-05
    AAPL: down=31.7%  stat=38.2%  up=30.1%
    INTC: down=9.1%  stat=82.4%  up=8.5%

  alpha=5e-05
    AAPL: down=14.8%  stat=70.6%  up=14.6%
    INTC: down=7.6%  stat=85.3%  up=7.1%

  alpha=1e-04
    AAPL: down=3.6%  stat=92.9%  up=3.5%
    INTC: down=5.1%  stat=90.3%  up=4.6%



In [4]:
def engineer_features(df: pd.DataFrame, n_levels: int = 10) -> pd.DataFrame:
    features = pd.DataFrame(index=df.index)

    # --- Group 1: Price features (10) ---
    # Spread and mid at each level
    for i in range(1, n_levels + 1):
        features[f"spread_{i}"] = df[f"ask_price_{i}"] - df[f"bid_price_{i}"]
    # Mid-price differences between levels (depth of book)
    for i in range(1, n_levels):
        features[f"ask_depth_{i}"] = df[f"ask_price_{i+1}"] - df[f"ask_price_{i}"]
        features[f"bid_depth_{i}"] = df[f"bid_price_{i}"] - df[f"bid_price_{i+1}"]

    # --- Group 2: Volume features (10) ---
    for i in range(1, n_levels + 1):
        features[f"ask_size_{i}"] = df[f"ask_size_{i}"]
        features[f"bid_size_{i}"] = df[f"bid_size_{i}"]

    # --- Group 3: Order imbalance at each level (10) ---
    for i in range(1, n_levels + 1):
        bid = df[f"bid_size_{i}"].astype(float)
        ask = df[f"ask_size_{i}"].astype(float)
        denom = bid + ask
        features[f"oi_{i}"] = np.where(denom > 0, (bid - ask) / denom, 0.0)

    # --- Group 4: Cumulative volume imbalance (5) ---
    # Measures pressure across multiple levels, not just best bid/ask
    for i in range(1, n_levels + 1):
        cum_bid = sum(df[f"bid_size_{j}"] for j in range(1, i + 1))
        cum_ask = sum(df[f"ask_size_{j}"] for j in range(1, i + 1))
        denom   = cum_bid + cum_ask
        features[f"cum_oi_{i}"] = np.where(denom > 0,
                                            (cum_bid - cum_ask) / denom, 0.0)

    # --- Group 5: Temporal / message features (5) ---
    features["event_type"]   = df["event_type"]
    features["trade_size"]   = df["size"]
    features["direction"]    = df["direction"]
    features["time_of_day"]  = df["time"] / 57600   # normalize to [0,1]
    features["mid_price"]    = df["mid_price"]       # kept for label alignment, dropped before training

    return features

# Test on AAPL
feats = engineer_features(data["AAPL"])
print(f"Feature matrix shape: {feats.shape}")
print(f"Columns: {list(feats.columns)}")
print(f"\nNull counts:\n{feats.isnull().sum()[feats.isnull().sum() > 0]}")

Feature matrix shape: (400391, 73)
Columns: ['spread_1', 'spread_2', 'spread_3', 'spread_4', 'spread_5', 'spread_6', 'spread_7', 'spread_8', 'spread_9', 'spread_10', 'ask_depth_1', 'bid_depth_1', 'ask_depth_2', 'bid_depth_2', 'ask_depth_3', 'bid_depth_3', 'ask_depth_4', 'bid_depth_4', 'ask_depth_5', 'bid_depth_5', 'ask_depth_6', 'bid_depth_6', 'ask_depth_7', 'bid_depth_7', 'ask_depth_8', 'bid_depth_8', 'ask_depth_9', 'bid_depth_9', 'ask_size_1', 'bid_size_1', 'ask_size_2', 'bid_size_2', 'ask_size_3', 'bid_size_3', 'ask_size_4', 'bid_size_4', 'ask_size_5', 'bid_size_5', 'ask_size_6', 'bid_size_6', 'ask_size_7', 'bid_size_7', 'ask_size_8', 'bid_size_8', 'ask_size_9', 'bid_size_9', 'ask_size_10', 'bid_size_10', 'oi_1', 'oi_2', 'oi_3', 'oi_4', 'oi_5', 'oi_6', 'oi_7', 'oi_8', 'oi_9', 'oi_10', 'cum_oi_1', 'cum_oi_2', 'cum_oi_3', 'cum_oi_4', 'cum_oi_5', 'cum_oi_6', 'cum_oi_7', 'cum_oi_8', 'cum_oi_9', 'cum_oi_10', 'event_type', 'trade_size', 'direction', 'time_of_day', 'mid_price']

Null count

In [5]:
feats_clean = feats.drop(columns=["mid_price"]).dropna()

corr = feats_clean.corr().abs()
# Find pairs with correlation > 0.95
high_corr = []
cols = corr.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        if corr.iloc[i, j] > 0.95:
            high_corr.append((cols[i], cols[j], corr.iloc[i, j]))

print(f"Highly correlated pairs (>0.95): {len(high_corr)}")
for a, b, c in sorted(high_corr, key=lambda x: -x[2])[:10]:
    print(f"  {a} <-> {b}: {c:.3f}")

Highly correlated pairs (>0.95): 12
  oi_1 <-> cum_oi_1: 1.000
  spread_9 <-> spread_10: 0.990
  spread_8 <-> spread_9: 0.989
  spread_7 <-> spread_8: 0.986
  spread_6 <-> spread_7: 0.980
  spread_8 <-> spread_10: 0.977
  spread_7 <-> spread_9: 0.972
  spread_5 <-> spread_6: 0.972
  spread_6 <-> spread_8: 0.962
  spread_4 <-> spread_5: 0.961


In [6]:
K     = 50
ALPHA = 1e-5

print(f"Final label distribution — k={K}, alpha={ALPHA}\n")
for ticker in ["AAPL", "AMZN", "GOOG"]:
    labels = make_labels(data[ticker]["mid_price"], k=K, alpha=ALPHA)
    counts = labels.value_counts().sort_index()
    total  = counts.sum()
    pcts   = (counts / total * 100).round(1)
    print(f"  {ticker}: down={pcts.get(0,0)}%  stat={pcts.get(1,0)}%  up={pcts.get(2,0)}%")

Final label distribution — k=50, alpha=1e-05

  AAPL: down=40.2%  stat=22.0%  up=37.9%
  AMZN: down=42.9%  stat=17.2%  up=39.9%
  GOOG: down=45.8%  stat=15.9%  up=38.3%


In [7]:
TICKERS_FINAL = ["AAPL", "AMZN", "GOOG"]
K             = 50
ALPHA         = 1e-5

def build_dataset(df, k=50, alpha=1e-5, n_levels=10):
    """
    Returns:
        X : np.ndarray of shape (N, n_features) — normalized features
        y : np.ndarray of shape (N,)             — integer labels {0,1,2}
    with NaN rows and tail rows (no valid label) removed.
    """
    feats  = engineer_features(df, n_levels)
    labels = make_labels(df["mid_price"], k=k, alpha=alpha)

    # Drop cum_oi_1 (identical to oi_1)
    feats  = feats.drop(columns=["cum_oi_1", "mid_price"])

    # Align and drop nulls
    combined        = feats.copy()
    combined["label"] = labels
    combined          = combined.dropna()

    # Drop tail rows where label horizon exceeds data
    combined = combined.iloc[:-k]

    X = combined.drop(columns=["label"]).values.astype(np.float32)
    y = combined["label"].values.astype(np.int64)

    return X, y

# Build and report
datasets = {}
for ticker in TICKERS_FINAL:
    X, y = build_dataset(data[ticker], k=K, alpha=ALPHA)
    datasets[ticker] = (X, y)
    counts = np.bincount(y)
    total  = len(y)
    print(f"{ticker}: X={X.shape}  y={y.shape}  "
          f"down={counts[0]/total*100:.1f}%  "
          f"stat={counts[1]/total*100:.1f}%  "
          f"up={counts[2]/total*100:.1f}%")

AAPL: X=(400341, 71)  y=(400341,)  down=40.2%  stat=22.0%  up=37.9%
AMZN: X=(269698, 71)  y=(269698,)  down=42.9%  stat=17.2%  up=39.9%
GOOG: X=(147866, 71)  y=(147866,)  down=45.9%  stat=15.8%  up=38.3%


In [8]:
def split_and_scale(X, y, train_frac=0.8, val_frac=0.1):
    """
    Sequential split preserving time order.
    Scaler fit on train only, applied to val and test.
    """
    n       = len(X)
    t1      = int(n * train_frac)
    t2      = int(n * (train_frac + val_frac))

    X_train, y_train = X[:t1],    y[:t1]
    X_val,   y_val   = X[t1:t2],  y[t1:t2]
    X_test,  y_test  = X[t2:],    y[t2:]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler

splits  = {}
scalers = {}
for ticker in TICKERS_FINAL:
    X, y = datasets[ticker]
    train, val, test, scaler = split_and_scale(X, y)
    splits[ticker]  = {"train": train, "val": val, "test": test}
    scalers[ticker] = scaler
    print(f"{ticker}: train={len(train[0])}  val={len(val[0])}  test={len(test[0])}")

AAPL: train=320272  val=40034  test=40035
AMZN: train=215758  val=26970  test=26970
GOOG: train=118292  val=14787  test=14787


In [9]:
class LOBDataset(Dataset):
    """
    Sliding window dataset for LOB sequence modeling.
    
    Each sample: (window of T snapshots, label at final timestep)
    X shape per sample: (T, n_features)
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, window: int = 100):
        self.X      = torch.tensor(X, dtype=torch.float32)
        self.y      = torch.tensor(y, dtype=torch.long)
        self.window = window

    def __len__(self):
        return len(self.X) - self.window

    def __getitem__(self, idx):
        x_seq = self.X[idx : idx + self.window]        # (T, n_features)
        label = self.y[idx + self.window - 1]           # scalar
        return x_seq, label

WINDOW     = 100
BATCH_SIZE = 512   # conservative for 4GB VRAM

loaders = {}
for ticker in TICKERS_FINAL:
    train_ds = LOBDataset(*splits[ticker]["train"], window=WINDOW)
    val_ds   = LOBDataset(*splits[ticker]["val"],   window=WINDOW)
    test_ds  = LOBDataset(*splits[ticker]["test"],  window=WINDOW)

    loaders[ticker] = {
        "train": DataLoader(train_ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0, pin_memory=True),
        "val":   DataLoader(val_ds,   batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0, pin_memory=True),
        "test":  DataLoader(test_ds,  batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0, pin_memory=True),
    }

    # Verify batch shape
    xb, yb = next(iter(loaders[ticker]["train"]))
    print(f"{ticker}: batch x={xb.shape}  y={yb.shape}")

AAPL: batch x=torch.Size([512, 100, 71])  y=torch.Size([512])
AMZN: batch x=torch.Size([512, 100, 71])  y=torch.Size([512])
GOOG: batch x=torch.Size([512, 100, 71])  y=torch.Size([512])


In [10]:
import pickle

save_path = PROCESSED_DIR / "lobster_processed.pkl"
payload   = {
    "splits":  splits,
    "scalers": scalers,
    "loaders": loaders,
    "config":  {"k": K, "alpha": ALPHA, "window": WINDOW,
                "batch_size": BATCH_SIZE, "tickers": TICKERS_FINAL}
}
with open(save_path, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved to {save_path}")

Saved to ../data/processed/lobster_processed.pkl
